In [29]:
from protossl.datasets import HeedbECGDataset
from protossl.defines import HEEDB_TARGETS

import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve, balanced_accuracy_score, roc_auc_score

dataset_path = "/opt/gpudata/ecg/heedb"

In [2]:
ds_phys = HeedbECGDataset(
    dataset_path=dataset_path,
    split="test",
    sampling_rate=100,
    label_src="original_physician",
)

ds_muse = HeedbECGDataset(
    dataset_path=dataset_path,
    split="test",
    sampling_rate=100,
    label_src="original_muse",
)

ds_muse_v24 = HeedbECGDataset(
    dataset_path=dataset_path,
    split="test",
    sampling_rate=100,
    label_src="new_muse",
)

================get_heedb_metadata=================
reading HEEDB metadata
read MGB data
read Emory data
=================make_heedb_labels=================


Converting code string to labels: 100%|██████████| 368191/368191 [00:03<00:00, 104052.12it/s]


Of 368191 ECGs and 11607259 annotations, 368191 matched (0 ECG annotations were missing and filled with 0s)
===============StreamingECGWaveforms===============
Using streaming ECG waveforms, will load and transform data on the fly
=================make_heedb_labels=================


Converting code string to labels: 100%|██████████| 368191/368191 [00:01<00:00, 242206.99it/s]


Of 368191 ECGs and 11607259 annotations, 368191 matched (0 ECG annotations were missing and filled with 0s)
===============StreamingECGWaveforms===============
Using streaming ECG waveforms, will load and transform data on the fly
=================make_heedb_labels=================


Converting code string to labels: 100%|██████████| 368191/368191 [00:05<00:00, 64363.89it/s] 


Of 368191 ECGs and 11445703 annotations, 368191 matched (0 ECG annotations were missing and filled with 0s)
===============StreamingECGWaveforms===============
Using streaming ECG waveforms, will load and transform data on the fly


In [3]:
y_true = ds_phys.labels.numpy()
y_pred_muse = ds_muse.labels.numpy()
y_pred_v24 = ds_muse_v24.labels.numpy()

In [4]:
def compute_pred(_y_true, _y_prob):
    fpr, tpr, thresholds = roc_curve(_y_true, _y_prob)
    j_scores = tpr - fpr
    optimal_idx = np.argmax(j_scores)
    optimal_threshold = thresholds[optimal_idx]
    return (_y_prob > optimal_threshold).astype(int)

In [31]:
y_prob_supproto = np.load("/opt/gpu_working/steven/protossl-ecg-outputs/supproto-heedb/probs.npy")
y_prob_protossl = np.load("/opt/gpu_working/steven/protossl-ecg-outputs/protossl-heedb/probs.npy")

y_pred_supproto = np.zeros_like(y_pred_muse)
y_pred_protossl = np.zeros_like(y_pred_muse)
results = []
for i, label in enumerate(HEEDB_TARGETS):
    print(label)
    y_pred_supproto[:, i] = compute_pred(y_true[:, i], y_prob_supproto[:, i])
    y_pred_protossl[:, i] = compute_pred(y_true[:, i], y_prob_protossl[:, i])
    label_results = {
        "label": label,
        # "supproto": balanced_accuracy_score(y_true[:, i], y_pred_supproto[:, i]),
        # "protossl": balanced_accuracy_score(y_true[:, i], y_pred_protossl[:, i]),
        # "old_muse": balanced_accuracy_score(y_true[:, i], y_pred_muse[:, i]),
        # "new_muse": balanced_accuracy_score(y_true[:, i], y_pred_v24[:, i]),
        "supproto": roc_auc_score(y_true[:, i], y_prob_supproto[:, i]),
        # "protossl": roc_auc_score(y_true[:, i], y_prob_protossl[:, i]),
    }
    results.append(label_results)
    print(f"supproto: {label_results['supproto']:0.3f}")
    # print(f"protossl: {label_results['protossl']:0.3f}")
    # print(f"old_muse: {label_results['old_muse']:0.3f}")
    # print(f"new_muse: {label_results['new_muse']:0.3f}")
    print()

ANTERIOR INFARCT
supproto: 0.797

ATRIAL FIBRILLATION
supproto: 0.962

ATRIAL FLUTTER
supproto: 0.947

ATRIAL-PACED RHYTHM
supproto: 0.988

INCOMPLETE RIGHT BUNDLE BRANCH BLOCK
supproto: 0.967

INFERIOR INFARCT
supproto: 0.834

LATERAL INFARCT
supproto: 0.945

LEFT BUNDLE BRANCH BLOCK
supproto: 0.988

NORMAL SINUS RHYTHM
supproto: 0.980

PREMATURE ATRIAL COMPLEXES
supproto: 0.880

PREMATURE VENTRICULAR COMPLEXES
supproto: 0.875

RIGHT AXIS DEVIATION
supproto: 0.987

RIGHT BUNDLE BRANCH BLOCK
supproto: 0.992

SINUS BRADYCARDIA
supproto: 0.994

SINUS RHYTHM
supproto: 0.885

SINUS TACHYCARDIA
supproto: 0.994

VENTRICULAR TACHYCARDIA
supproto: 0.895

VENTRICULAR-PACED RHYTHM
supproto: 0.991

WITH 1ST DEGREE AV BLOCK
supproto: 0.988

WITH SINUS ARRHYTHMIA
supproto: 0.943



In [32]:
print(pd.DataFrame(results).to_string(float_format="%0.3f", index=False))

                               label  supproto
                    ANTERIOR INFARCT     0.797
                 ATRIAL FIBRILLATION     0.962
                      ATRIAL FLUTTER     0.947
                 ATRIAL-PACED RHYTHM     0.988
INCOMPLETE RIGHT BUNDLE BRANCH BLOCK     0.967
                    INFERIOR INFARCT     0.834
                     LATERAL INFARCT     0.945
            LEFT BUNDLE BRANCH BLOCK     0.988
                 NORMAL SINUS RHYTHM     0.980
          PREMATURE ATRIAL COMPLEXES     0.880
     PREMATURE VENTRICULAR COMPLEXES     0.875
                RIGHT AXIS DEVIATION     0.987
           RIGHT BUNDLE BRANCH BLOCK     0.992
                   SINUS BRADYCARDIA     0.994
                        SINUS RHYTHM     0.885
                   SINUS TACHYCARDIA     0.994
             VENTRICULAR TACHYCARDIA     0.895
            VENTRICULAR-PACED RHYTHM     0.991
            WITH 1ST DEGREE AV BLOCK     0.988
               WITH SINUS ARRHYTHMIA     0.943


In [33]:
with pd.option_context("display.precision", 3):
    display(pd.DataFrame(results))

,label,supproto
0,ANTERIOR INFARCT,0.797
1,ATRIAL FIBRILLATION,0.962
2,ATRIAL FLUTTER,0.947
3,ATRIAL-PACED RHYTHM,0.988
4,INCOMPLETE RIGHT BUNDLE BRANCH BLOCK,0.967
5,INFERIOR INFARCT,0.834
6,LATERAL INFARCT,0.945
7,LEFT BUNDLE BRANCH BLOCK,0.988
8,NORMAL SINUS RHYTHM,0.980
9,PREMATURE ATRIAL COMPLEXES,0.880


In [34]:
pd.DataFrame(results).set_index("label").mean(axis=0)

supproto    0.941648
dtype: float64

In [21]:
ds_phys.labels[:, 0].sum()

tensor(37)

In [19]:
ds_muse.labels[:, 0].sum()

tensor(0)

In [20]:
ds_muse_v24.labels[:, 0].sum()

tensor(21096)

In [22]:
ds_phys.labels.shape

torch.Size([368191, 20])

In [23]:
ds_train_phys = HeedbECGDataset(
    dataset_path=dataset_path,
    split="train",
    sampling_rate=100,
    label_src="original_physician",
)

=================make_heedb_labels=================


Converting code string to labels: 100%|██████████| 8145485/8145485 [01:33<00:00, 87111.97it/s] 


Of 8145485 ECGs and 11607259 annotations, 8145484 matched (1 ECG annotations were missing and filled with 0s)
===============StreamingECGWaveforms===============
Using streaming ECG waveforms, will load and transform data on the fly


In [24]:
ds_valid_phys = HeedbECGDataset(
    dataset_path=dataset_path,
    split="val",
    sampling_rate=100,
    label_src="original_physician",
)

=================make_heedb_labels=================


Converting code string to labels: 100%|██████████| 556578/556578 [00:01<00:00, 512100.29it/s]


Of 556578 ECGs and 11607259 annotations, 556578 matched (0 ECG annotations were missing and filled with 0s)
===============StreamingECGWaveforms===============
Using streaming ECG waveforms, will load and transform data on the fly


In [26]:
ds_train_phys.labels[:, 0].sum()

tensor(100103)

In [27]:
ds_train_phys.labels.shape[0]

8145485

In [28]:
ds_train_phys.labels[:, 0].sum() / ds_train_phys.labels.shape[0]

tensor(0.0123)

In [1]:
368191 + 556578 + 8145485

9070254